In [1]:
import pandas as pd
from pathlib import Path
import altair as alt

In [2]:
df = pd.read_csv('website/cleaned_listings.csv')

In [3]:
# filter the data to exclude outliers
Q1 = df['minimum_nights'].quantile(0.25)
Q3 = df['minimum_nights'].quantile(0.75)
IQR = Q3 - Q1

threshold = Q3 + 1.5 * IQR
df_filtered = df[(df['minimum_nights'] <= threshold)].copy()

_, bin_edges = pd.qcut(df['availability_365'], q=4, retbins=True, duplicates='drop')

categorical_order = []

prefixes = ['Low', 'Medium', 'High', 'Very High']

for i in range(len(bin_edges)-1):
    lower = int(bin_edges[i])
    upper = int(bin_edges[i+1])
    
    if len(bin_edges)-1 == 4:
        label = f"{prefixes[i]} ({lower}-{upper})"
    else:
        label = f"{lower}-{upper}"
    categorical_order.append(label)

df_filtered['availability_category'] = pd.cut(
    df_filtered['availability_365'], 
    bins=bin_edges,
    labels=categorical_order,
    include_lowest=True
)

In [4]:
color_order = ['null'] + categorical_order

order_map = {cat: i for i, cat in enumerate(color_order)}

df_filtered = df_filtered.copy()
df_filtered['availability_order'] = df_filtered['availability_category'].map(order_map)

In [ ]:
color_order = ['null'] + categorical_order

chart1 = alt.Chart(df_filtered).mark_bar(opacity=0.7,  stroke=None   ).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(step=2),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis', domain=color_order), sort=color_order),
    order=alt.Order('availability_order:Q'),
    tooltip=[
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()

c1_json = chart1.to_json()

with open('website/t4-barplot1_spec.json', 'w') as f:
    f.write(c1_json)

